# YOLO12-Small + GhostConv — CH-RDD2022 (Kaggle)

Notebook ablation ini memakai **GhostConv saja**, tanpa ECA, EMA, SPD-Conv, atau attention lain. GhostConv menggantikan downsampling P4/16 dan P5/32; jalur P3 tetap memakai Conv standar agar detail cacat kecil terjaga.

Notebook meng-clone branch `yolo12-ghost-conv`, memasangnya editable, mentransfer bobot `yolo12s.pt` yang kompatibel, lalu mengarsipkan hasil ke ZIP. Aktifkan GPU dan Internet pada Kaggle.

In [ ]:
# 1. Clone branch modifikasi dan install repository secara editable.
import json
import platform
import subprocess
import sys
import zipfile
from pathlib import Path

WORKDIR = Path('/kaggle/working')
REPO_URL = 'https://github.com/danial2015/yolo-aceh-rdd2022.git'
REPO_BRANCH = 'yolo12-ghost-conv'
REPO_DIR = WORKDIR / 'yolo-aceh-rdd2022'

def log_section(title: str) -> None:
    print(f'\n{"=" * 90}\n{title}\n{"=" * 90}')

log_section('CLONE AND INSTALL GHOSTCONV REPOSITORY')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--depth', '1', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--force', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', f'origin/{REPO_BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)

REPO_COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
REPO_METADATA = WORKDIR / 'repository_revision.txt'
REPO_METADATA.write_text(f'repository={REPO_URL}\nbranch={REPO_BRANCH}\ncommit={REPO_COMMIT}\n', encoding='utf-8')
sys.path.insert(0, str(REPO_DIR))
import torch
import ultralytics

log_section('ENVIRONMENT')
print(f'Python      : {platform.python_version()}')
print(f'PyTorch     : {torch.__version__}')
print(f'Ultralytics : {ultralytics.__version__}')
print(f'Commit      : {REPO_COMMIT}')
print(f'CUDA ready  : {torch.cuda.is_available()}')
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
if torch.cuda.is_available():
    print(f'GPU         : {torch.cuda.get_device_name(0)}')

In [ ]:
# 2. Dataset, hyperparameter, dan verifikasi arsitektur GhostConv-only.
DATA_ROOT = Path('/kaggle/input/datasets/danialalfayyadh/ch-rdd2022/datasets-china-split')
DATA_YAML = WORKDIR / 'ch_rdd2022.yaml'
MODEL_YAML = REPO_DIR / 'ultralytics/cfg/models/12/yolo12-ghost.yaml'
CUSTOM_SOURCE_FILES = (REPO_DIR / 'ultralytics/nn/modules/conv.py', MODEL_YAML)

# Samakan setting ini dengan baseline dan semua varian attention untuk ablation yang fair.
EPOCHS, IMGSZ, BATCH = 160, 640, 64
OPTIMIZER, LR0, MOMENTUM, WEIGHT_DECAY = 'SGD', 0.01, 0.937, 0.0005
PATIENCE, WORKERS, SEED = 0, 2, 42
EXPERIMENT_NAME = 'yolo12s_ghost_ch_rdd2022_pretrained'
RUNS_DIR = WORKDIR / 'runs'

DATA_YAML.write_text(
    f'''path: {DATA_ROOT}
train: train/images
val: val/images
test: test/images

nc: 5
names:
  0: D00
  1: D10
  2: D20
  3: D40
  4: Repair
''',
    encoding='utf-8',
)
assert DATA_ROOT.exists(), f'Dataset path tidak ditemukan: {DATA_ROOT}'
assert MODEL_YAML.exists(), f'Model YAML tidak ditemukan: {MODEL_YAML}'

from ultralytics.nn.modules import GhostConv
from ultralytics.nn.tasks import DetectionModel

log_section('YOLO12S + GHOSTCONV-ONLY MODEL INFO')
print(MODEL_YAML.read_text(encoding='utf-8'))
check_model = DetectionModel(str(MODEL_YAML), nc=5, verbose=False)
parameter_count = sum(parameter.numel() for parameter in check_model.parameters())
ghost_layers = [layer.i for layer in check_model.model if isinstance(layer, GhostConv)]
attention_layers = [layer for layer in check_model.model if 'Attention' in layer.__class__.__name__]
assert ghost_layers == [5, 7], f'GhostConv tidak terbentuk sesuai konfigurasi: {ghost_layers}'
assert not attention_layers, f'Ghost-only tidak boleh mempunyai attention module: {attention_layers}'
print(f'Parameters (5 classes): {parameter_count:,}')
print(f'GhostConv layer indices : {ghost_layers}')
print('Attention layers        : none')
check_model.info(detailed=False, verbose=True)
del check_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# 3. Transfer bobot pretrained YOLO12s yang tetap kompatibel dengan GhostConv-only.
from ultralytics import YOLO

PRETRAINED_WEIGHTS = 'yolo12s.pt'

def compatible_yolo12s_weights(source_state: dict, target_state: dict) -> dict:
    """Transfer tensor dengan nama dan bentuk sama; GhostConv dan head baru akan dipelajari saat fine-tuning."""
    return {
        key: tensor for key, tensor in source_state.items()
        if key in target_state and target_state[key].shape == tensor.shape
    }

log_section('PRETRAINED WEIGHT TRANSFER')
model = YOLO(str(MODEL_YAML))
source_model = YOLO(PRETRAINED_WEIGHTS).model.float()
target_state = model.model.state_dict()
transferred_state = compatible_yolo12s_weights(source_model.state_dict(), target_state)
incompatible = model.model.load_state_dict(transferred_state, strict=False)
PRETRAINED_REPORT = {
    'source_weights': PRETRAINED_WEIGHTS,
    'transferred_tensors': len(transferred_state),
    'target_tensors': len(target_state),
    'uninitialized_tensors': len(incompatible.missing_keys),
    'uninitialized_tensor_names': incompatible.missing_keys,
}
model.ckpt = {'model': model.model}
del source_model, target_state, transferred_state
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"Transferred tensors : {PRETRAINED_REPORT['transferred_tensors']}/{PRETRAINED_REPORT['target_tensors']}")
print(f"Uninitialized tensors: {PRETRAINED_REPORT['uninitialized_tensors']} (GhostConv dan head 5 kelas)")
print('Model melakukan fine-tuning dari yolo12s.pt, bukan training dari nol.')

In [ ]:
# 4. Training GhostConv-only.
log_section('TRAINING STARTED — GHOSTCONV ONLY')
print(f'epochs={EPOCHS}, imgsz={IMGSZ}, batch={BATCH}, optimizer={OPTIMIZER}, lr0={LR0}, seed={SEED}')
model.train(
    data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=DEVICE, workers=WORKERS,
    project=str(RUNS_DIR), name=EXPERIMENT_NAME, exist_ok=True, pretrained=True, optimizer=OPTIMIZER,
    lr0=LR0, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY, cos_lr=False, patience=PATIENCE,
    seed=SEED, plots=True, verbose=True,
)
RUN_DIR, BEST_PT, LAST_PT = Path(model.trainer.save_dir), Path(model.trainer.best), Path(model.trainer.last)
print(f'Run directory: {RUN_DIR}')
print(f'Best weights : {BEST_PT}')
print(f'Last weights : {LAST_PT}')

In [ ]:
# 5. Evaluasi best.pt, arsipkan metrik, checkpoint, YAML, dan kode ke ZIP.
log_section('BEST CHECKPOINT EVALUATION')
best_model = YOLO(str(BEST_PT))

def metric_summary(metrics) -> dict:
    return {
        'precision': float(metrics.box.mp), 'recall': float(metrics.box.mr),
        'map50': float(metrics.box.map50), 'map50_95': float(metrics.box.map),
        'save_dir': str(metrics.save_dir),
    }

val_metrics = best_model.val(
    data=str(DATA_YAML), split='val', imgsz=IMGSZ, batch=16, device=DEVICE, project=str(RUNS_DIR),
    name=f'{EXPERIMENT_NAME}_val', exist_ok=True, plots=True,
)
EVALUATION_REPORT = {'validation': metric_summary(val_metrics)}
test_label_dir = DATA_ROOT / 'test' / 'labels'
if test_label_dir.exists() and any(test_label_dir.glob('*.txt')):
    test_metrics = best_model.val(
        data=str(DATA_YAML), split='test', imgsz=IMGSZ, batch=16, device=DEVICE, project=str(RUNS_DIR),
        name=f'{EXPERIMENT_NAME}_test', exist_ok=True, plots=True,
    )
    EVALUATION_REPORT['test'] = metric_summary(test_metrics)
    TEST_OUTPUT_DIR = Path(test_metrics.save_dir)
else:
    predictions = best_model.predict(
        source=str(DATA_ROOT / 'test' / 'images'), imgsz=IMGSZ, device=DEVICE, conf=0.25, save=True,
        save_txt=True, project=str(RUNS_DIR), name=f'{EXPERIMENT_NAME}_test_predictions', exist_ok=True,
    )
    TEST_OUTPUT_DIR = Path(predictions[0].save_dir) if predictions else RUNS_DIR
    EVALUATION_REPORT['test'] = {'status': 'labels unavailable; prediction only', 'save_dir': str(TEST_OUTPUT_DIR)}

EVALUATION_JSON = WORKDIR / f'{EXPERIMENT_NAME}_evaluation_metrics.json'
EVALUATION_JSON.write_text(json.dumps(EVALUATION_REPORT, indent=2), encoding='utf-8')
print(json.dumps(EVALUATION_REPORT, indent=2))

RUN_CONFIG = WORKDIR / f'{EXPERIMENT_NAME}_config.json'
RUN_CONFIG.write_text(json.dumps({
    'dataset_root': str(DATA_ROOT), 'repository_url': REPO_URL, 'repository_branch': REPO_BRANCH,
    'repository_commit': REPO_COMMIT, 'model_yaml': str(MODEL_YAML),
    'ghost_conv_positions': 'P4/16 and P5/32 downsampling only; no attention module',
    'pretrained_transfer': PRETRAINED_REPORT, 'epochs': EPOCHS, 'imgsz': IMGSZ, 'batch': BATCH,
    'optimizer': OPTIMIZER, 'lr0': LR0, 'momentum': MOMENTUM, 'weight_decay': WEIGHT_DECAY,
    'seed': SEED, 'best_checkpoint': str(BEST_PT), 'last_checkpoint': str(LAST_PT),
}, indent=2), encoding='utf-8')

ZIP_PATH = WORKDIR / f'{EXPERIMENT_NAME}_results.zip'
def add_to_zip(archive: zipfile.ZipFile, path: Path) -> int:
    if not path.exists():
        return 0
    files = [path] if path.is_file() else [item for item in path.rglob('*') if item.is_file()]
    for file_path in files:
        archive.write(file_path, file_path.relative_to(WORKDIR))
    return len(files)

with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    count = sum(add_to_zip(archive, Path(item)) for item in (
        RUN_DIR, Path(val_metrics.save_dir), TEST_OUTPUT_DIR, DATA_YAML, *CUSTOM_SOURCE_FILES,
        REPO_METADATA, RUN_CONFIG, EVALUATION_JSON,
    ))
print(f'ZIP created : {ZIP_PATH}')
print(f'Files added : {count}')
from IPython.display import FileLink, display
display(FileLink(ZIP_PATH))